In [5]:
import torch
from base import BaseVAE
from torch import nn
from torch.nn import functional as F
from types_ import *

In [2]:
import importlib

module = importlib.import_module("tabpfn")
version = getattr(module, "__version__")
print(version)

6.3.1


In [6]:
# import torch
# from models import BaseVAE
# from torch import nn
# from torch.nn import functional as F
# from .types_ import *


class BetaVAE(BaseVAE):

    num_iter = 0 # Global static variable to keep track of iterations

    def __init__(self,
                 in_channels: int,
                 latent_dim: int,
                 hidden_dims: List = None,
                 beta: int = 4,
                 gamma:float = 1000.,
                 max_capacity: int = 25,
                 Capacity_max_iter: int = 1e5,
                 loss_type:str = 'B',
                 **kwargs) -> None:
        super(BetaVAE, self).__init__()

        self.latent_dim = latent_dim
        self.beta = beta
        self.gamma = gamma
        self.loss_type = loss_type
        self.C_max = torch.Tensor([max_capacity])
        self.C_stop_iter = Capacity_max_iter

        modules = []
        if hidden_dims is None:
            hidden_dims = [32, 64, 128, 256, 512]

        # Build Encoder
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, out_channels=h_dim,
                              kernel_size= 3, stride= 2, padding  = 1),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU())
            )
            in_channels = h_dim

        self.encoder = nn.Sequential(*modules)
        self.fc_mu = nn.Linear(hidden_dims[-1]*4, latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1]*4, latent_dim)


        # Build Decoder
        modules = []

        self.decoder_input = nn.Linear(latent_dim, hidden_dims[-1] * 4)

        hidden_dims.reverse()

        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i],
                                       hidden_dims[i + 1],
                                       kernel_size=3,
                                       stride = 2,
                                       padding=1,
                                       output_padding=1),
                    nn.BatchNorm2d(hidden_dims[i + 1]),
                    nn.LeakyReLU())
            )



        self.decoder = nn.Sequential(*modules)

        self.final_layer = nn.Sequential(
                            nn.ConvTranspose2d(hidden_dims[-1],
                                               hidden_dims[-1],
                                               kernel_size=3,
                                               stride=2,
                                               padding=1,
                                               output_padding=1),
                            nn.BatchNorm2d(hidden_dims[-1]),
                            nn.LeakyReLU(),
                            nn.Conv2d(hidden_dims[-1], out_channels= 3,
                                      kernel_size= 3, padding= 1),
                            nn.Tanh())#输出在 [-1,1]，说明输入图像必须也归一化到 [-1,1]

    def encode(self, input: Tensor) -> List[Tensor]:
        """
        Encodes the input by passing through the encoder network
        and returns the latent codes.
        :param input: (Tensor) Input tensor to encoder [N x C x H x W]
        :return: (Tensor) List of latent codes
        """
        result = self.encoder(input)
        result = torch.flatten(result, start_dim=1)

        # Split the result into mu and var components
        # of the latent Gaussian distribution
        mu = self.fc_mu(result)
        log_var = self.fc_var(result)

        return [mu, log_var]

    def decode(self, z: Tensor) -> Tensor:
        result = self.decoder_input(z)
        result = result.view(-1, 512, 2, 2)
        result = self.decoder(result)
        result = self.final_layer(result)
        return result

    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        """
        Will a single z be enough ti compute the expectation
        for the loss??
        :param mu: (Tensor) Mean of the latent Gaussian
        :param logvar: (Tensor) Standard deviation of the latent Gaussian
        :return:
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, input: Tensor, **kwargs) -> Tensor:
        mu, log_var = self.encode(input)
        z = self.reparameterize(mu, log_var)
        return  [self.decode(z), input, mu, log_var]

    def loss_function(self,
                      *args,
                      **kwargs) -> dict:
        self.num_iter += 1
        recons = args[0]
        input = args[1]
        mu = args[2]
        log_var = args[3]
        kld_weight = kwargs['M_N']  # Account for the minibatch samples from the dataset

        recons_loss =F.mse_loss(recons, input)

        kld_loss = torch.mean(-0.5 * torch.sum(1 + log_var - mu ** 2 - log_var.exp(), dim = 1), dim = 0)

        if self.loss_type == 'H': # BETA-VAE                    # https://openreview.net/forum?id=Sy2fzU9gl
            loss = recons_loss + self.beta * kld_weight * kld_loss
        elif self.loss_type == 'B': # Capacity-VAE    # https://arxiv.org/pdf/1804.03599.pdf
            self.C_max = self.C_max.to(input.device)
            C = torch.clamp(self.C_max/self.C_stop_iter * self.num_iter, 0, self.C_max.data[0])
            loss = recons_loss + self.gamma * kld_weight* (kld_loss - C).abs()
        else:
            raise ValueError('Undefined loss type.')

        return {'loss': loss, 'Reconstruction_Loss':recons_loss, 'KLD':kld_loss}

    def sample(self,
               num_samples:int,
               current_device: int, **kwargs) -> Tensor:
        """
        Samples from the latent space and return the corresponding
        image space map.
        :param num_samples: (Int) Number of samples
        :param current_device: (Int) Device to run the model
        :return: (Tensor)
        纯生成新样本（数据增强）
        """
        z = torch.randn(num_samples,
                        self.latent_dim)

        z = z.to(current_device)

        samples = self.decode(z)
        return samples

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        """
        Given an input image x, returns the reconstructed image
        :param x: (Tensor) [B x C x H x W]
        :return: (Tensor) [B x C x H x W]
        重构 / 编码质量评估
        """

        return self.forward(x)[0]

In [ ]:
class VanillaVAE(BaseVAE):
# 标准化VAE

    def __init__(self,
                 in_channels: int,
                 latent_dim: int,
                 hidden_dims: List = None,
                 **kwargs) -> None:
        super(VanillaVAE, self).__init__()

        self.latent_dim = latent_dim

        modules = []
        if hidden_dims is None:
            hidden_dims = [32, 64, 128, 256, 512]

        # Build Encoder
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, out_channels=h_dim,
                              kernel_size= 3, stride= 2, padding  = 1),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU())
            )
            in_channels = h_dim

        self.encoder = nn.Sequential(*modules)
        self.fc_mu = nn.Linear(hidden_dims[-1]*4, latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1]*4, latent_dim)


        # Build Decoder
        modules = []

        self.decoder_input = nn.Linear(latent_dim, hidden_dims[-1] * 4)

        hidden_dims.reverse()

        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i],
                                       hidden_dims[i + 1],
                                       kernel_size=3,
                                       stride = 2,
                                       padding=1,
                                       output_padding=1),
                    nn.BatchNorm2d(hidden_dims[i + 1]),
                    nn.LeakyReLU())
            )



        self.decoder = nn.Sequential(*modules)

        self.final_layer = nn.Sequential(
                            nn.ConvTranspose2d(hidden_dims[-1],
                                               hidden_dims[-1],
                                               kernel_size=3,
                                               stride=2,
                                               padding=1,
                                               output_padding=1),
                            nn.BatchNorm2d(hidden_dims[-1]),
                            nn.LeakyReLU(),
                            nn.Conv2d(hidden_dims[-1], out_channels= 3,
                                      kernel_size= 3, padding= 1),
                            nn.Tanh())

    def encode(self, input: Tensor) -> List[Tensor]:
        """
        Encodes the input by passing through the encoder network
        and returns the latent codes.
        :param input: (Tensor) Input tensor to encoder [N x C x H x W]
        :return: (Tensor) List of latent codes
        """
        result = self.encoder(input)
        result = torch.flatten(result, start_dim=1)

        # Split the result into mu and var components
        # of the latent Gaussian distribution
        mu = self.fc_mu(result)
        log_var = self.fc_var(result)

        return [mu, log_var]

    def decode(self, z: Tensor) -> Tensor:
        """
        Maps the given latent codes
        onto the image space.
        :param z: (Tensor) [B x D]
        :return: (Tensor) [B x C x H x W]
        """
        result = self.decoder_input(z)
        result = result.view(-1, 512, 2, 2)
        result = self.decoder(result)
        result = self.final_layer(result)
        return result

    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        """
        Reparameterization trick to sample from N(mu, var) from
        N(0,1).
        :param mu: (Tensor) Mean of the latent Gaussian [B x D]
        :param logvar: (Tensor) Standard deviation of the latent Gaussian [B x D]
        :return: (Tensor) [B x D]
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, input: Tensor, **kwargs) -> List[Tensor]:
        mu, log_var = self.encode(input)
        z = self.reparameterize(mu, log_var)
        return  [self.decode(z), input, mu, log_var]

    def loss_function(self,
                      *args,
                      **kwargs) -> dict:
        """
        Computes the VAE loss function.
        KL(N(\mu, \sigma), N(0, 1)) = \log \frac{1}{\sigma} + \frac{\sigma^2 + \mu^2}{2} - \frac{1}{2}
        :param args:
        :param kwargs:
        :return:
        """
        recons = args[0]
        input = args[1]
        mu = args[2]
        log_var = args[3]

        kld_weight = kwargs['M_N'] # Account for the minibatch samples from the dataset
        recons_loss =F.mse_loss(recons, input)


        kld_loss = torch.mean(-0.5 * torch.sum(1 + log_var - mu ** 2 - log_var.exp(), dim = 1), dim = 0)

        loss = recons_loss + kld_weight * kld_loss
        return {'loss': loss, 'Reconstruction_Loss':recons_loss.detach(), 'KLD':-kld_loss.detach()}

    def sample(self,
               num_samples:int,
               current_device: int, **kwargs) -> Tensor:
        """
        Samples from the latent space and return the corresponding
        image space map.
        :param num_samples: (Int) Number of samples
        :param current_device: (Int) Device to run the model
        :return: (Tensor)
        """
        z = torch.randn(num_samples,
                        self.latent_dim)

        z = z.to(current_device)

        samples = self.decode(z)
        return samples

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        """
        Given an input image x, returns the reconstructed image
        :param x: (Tensor) [B x C x H x W]
        :return: (Tensor) [B x C x H x W]
        """

        return self.forward(x)[0]

### linear vae


In [14]:
# from re import M
# import torch
# from torch import nn, relu
# from torch.distributions.kl import kl_divergence
# from torch.utils.data import DataLoader
# from torch.distributions.multivariate_normal import MultivariateNormal
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler


In [ ]:
# from re import M
# import torch
# from torch import nn, relu
# from torch.distributions.kl import kl_divergence
# from torch.distributions.multivariate_normal import MultivariateNormal


class LinearVariationalEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(LinearVariationalEncoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.nn_mean = nn.Linear(input_dim, latent_dim, bias=False)
        self.nn_logvar =  nn.Linear(input_dim, latent_dim, bias=True)
    
    def forward(self, x):
        batch_size = x.size(0)
        eps = torch.randn(batch_size, self.latent_dim)
        mu = self.nn_mean(x)
        logvar = self.nn_logvar(x)
        sigma = logvar.div(2).exp()
        return {'z': mu + sigma * eps,
                'mu': mu,
                'sigma': sigma}

class LinearVariationalDecoder(nn.Module):
    def __init__(self, latent_dim, target_dim):
        super(LinearVariationalDecoder, self).__init__()
        self.latent_dim = latent_dim
        self.target_dim = target_dim
        self.l = nn.Linear(latent_dim, target_dim, bias=False)

    def forward(self, z):
        y = self.l(z)
        return y

class LinearBetaVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, target_dim, eta_dec_sq, eta_prior_sq, beta, **kwargs):
        super(LinearBetaVAE, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.target_dim = target_dim
        self.eta_dec_sq = eta_dec_sq
        self.eta_prior_sq = eta_prior_sq
        self.beta = beta

        self.encoder = LinearVariationalEncoder(input_dim, latent_dim)
        self.decoder = LinearVariationalDecoder(latent_dim, target_dim)

    def forward(self, x, y):
        encoded = self.encoder(x)
        y_pred = self.decoder(encoded['z'])
        rec_loss = torch.square(y - y_pred).sum(-1).mean(0) / self.eta_dec_sq / 2

        kl_loss = .5 * (
            - torch.log(encoded['sigma']**2 / self.eta_prior_sq).sum(-1) 
            - self.latent_dim
            + torch.norm(encoded['mu'], p=2, dim=-1) ** 2 / self.eta_prior_sq
            + (encoded['sigma'] ** 2 / self.eta_prior_sq).sum(-1)
            ).mean(0)
        loss = rec_loss + self.beta * kl_loss
        forward_dict = {
            'z': encoded['z'],
            'mu': encoded['mu'],
            'sigma': encoded['sigma'],
            'y_pred': y_pred,
            'loss': loss,
            'rec_loss': rec_loss,
            'kl_loss': kl_loss * self.beta,
            'enc_norm': self.encoder.nn_mean.weight.norm(),
            'dec_norm': self.decoder.l.weight.norm()
        }

        return forward_dict


class ReLUVariationalEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim):
        super(ReLUVariationalEncoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.nn_mean = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
            )
        self.nn_logvar =  nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=True),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        eps = torch.randn(batch_size, self.latent_dim)
        mu = self.nn_mean(x)
        logvar = self.nn_logvar(x)
        sigma = logvar.div(2).exp()
        return {'z': mu + sigma * eps,
                'mu': mu,
                'sigma': sigma}

class ReLUVariationalDecoder(nn.Module):
    def __init__(self, latent_dim, target_dim, hidden_dim):
        super(ReLUVariationalDecoder, self).__init__()
        self.latent_dim = latent_dim
        self.target_dim = target_dim
        self.hidden_dim = hidden_dim
        self.l = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(hidden_dim, target_dim)
        )

    def forward(self, z):
        y = self.l(z)
        return y

class ReLUBetaVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, target_dim, hidden_dim, eta_dec_sq, eta_prior_sq, beta, **kwargs):
        super(ReLUBetaVAE, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.target_dim = target_dim
        self.hidden_dim = hidden_dim
        self.eta_dec_sq = eta_dec_sq
        self.eta_prior_sq = eta_prior_sq
        self.beta = beta

        self.encoder = ReLUVariationalEncoder(input_dim, latent_dim, hidden_dim)
        self.decoder = ReLUVariationalDecoder(latent_dim, target_dim, hidden_dim)

    def forward(self, x, y):
        encoded = self.encoder(x)
        y_pred = self.decoder(encoded['z'])
        rec_loss = torch.square(y - y_pred).sum(-1).mean(0) / self.eta_dec_sq / 2

        kl_loss = .5 * (
            - torch.log(encoded['sigma']**2 / self.eta_prior_sq).sum(-1) 
            - self.latent_dim
            + torch.norm(encoded['mu'], p=2, dim=-1) ** 2 / self.eta_prior_sq
            + (encoded['sigma'] ** 2 / self.eta_prior_sq).sum(-1)
            ).mean(0)
        loss = rec_loss + self.beta * kl_loss
        forward_dict = {
            'z': encoded['z'],
            'mu': encoded['mu'],
            'sigma': encoded['sigma'],
            'y_pred': y_pred,
            'loss': loss,
            'rec_loss': rec_loss,
            'kl_loss': kl_loss * self.beta,
            # 'enc_norm': self.encoder.nn_mean.weight.norm(),
            # 'dec_norm': self.decoder.l.weight.norm()
        }

        return forward_dict



class TanhVariationalEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim):
        super(TanhVariationalEncoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.nn_mean = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=False),
            nn.Tanh(),#nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
            )
        self.nn_logvar =  nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=True),
            nn.Tanh(),#nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        eps = torch.randn(batch_size, self.latent_dim)
        mu = self.nn_mean(x)
        logvar = self.nn_logvar(x)
        sigma = logvar.div(2).exp()
        return {'z': mu + sigma * eps,
                'mu': mu,
                'sigma': sigma}

class TanhVariationalDecoder(nn.Module):
    def __init__(self, latent_dim, target_dim, hidden_dim):
        super(TanhVariationalDecoder, self).__init__()
        self.latent_dim = latent_dim
        self.target_dim = target_dim
        self.hidden_dim = hidden_dim
        self.l = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(hidden_dim, target_dim)
        )

    def forward(self, z):
        y = self.l(z)
        return y

# class TanhBetaVAE(nn.Module):
#     def __init__(self, input_dim, latent_dim, target_dim, hidden_dim, eta_dec_sq, eta_prior_sq, beta, **kwargs):
#         super(TanhBetaVAE, self).__init__()
#         self.input_dim = input_dim
#         self.latent_dim = latent_dim
#         self.target_dim = target_dim
#         self.hidden_dim = hidden_dim
#         self.eta_dec_sq = eta_dec_sq
#         self.eta_prior_sq = eta_prior_sq
#         self.beta = beta

#         self.encoder = TanhVariationalEncoder(input_dim, latent_dim, hidden_dim)
#         self.decoder = TanhVariationalDecoder(latent_dim, target_dim, hidden_dim)

#     def forward(self, x, y):
#         encoded = self.encoder(x)
#         y_pred = self.decoder(encoded['z'])
#         rec_loss = torch.square(y - y_pred).sum(-1).mean(0) / self.eta_dec_sq / 2

#         kl_loss = .5 * (
#             - torch.log(encoded['sigma']**2 / self.eta_prior_sq).sum(-1) 
#             - self.latent_dim
#             + torch.norm(encoded['mu'], p=2, dim=-1) ** 2 / self.eta_prior_sq
#             + (encoded['sigma'] ** 2 / self.eta_prior_sq).sum(-1)
#             ).mean(0)
#         loss = rec_loss + self.beta * kl_loss
#         forward_dict = {
#             'z': encoded['z'],
#             'mu': encoded['mu'],
#             'sigma': encoded['sigma'],
#             'y_pred': y_pred,
#             'loss': loss,
#             'rec_loss': rec_loss,
        #     'kl_loss': kl_loss * self.beta,
        #     # 'enc_norm': self.encoder.nn_mean.weight.norm(),
        #     # 'dec_norm': self.decoder.l.weight.norm()
        # }

        # return forward_dict

In [15]:
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

# -------------------------------
# 1. 准备数据
# -------------------------------
data = load_breast_cancer()
X = data['data']  # [n_samples, n_features]
X = StandardScaler().fit_transform(X)  # 标准化

X_tensor = torch.tensor(X, dtype=torch.float32)

dataset = TensorDataset(X_tensor, X_tensor)  # VAE 输入=输出
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

input_dim = X_tensor.shape[1]
latent_dim = 5  # 潜变量维度

# -------------------------------
# 2. 定义 LinearVAE
# -------------------------------
class LinearVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.fc_mu = nn.Linear(input_dim, latent_dim)
        self.fc_logvar = nn.Linear(input_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, input_dim)

    def encode(self, x):
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.fc_dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

# -------------------------------
# 3. 定义损失函数
# -------------------------------
def vae_loss(x, x_hat, mu, logvar):
    # 重构误差 (MSE)
    recon_loss = nn.MSELoss()(x_hat, x)
    # KL 散度
    kld_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kld_loss, recon_loss, kld_loss

# -------------------------------
# 4. 训练模型
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinearVAE(input_dim, latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 50
for epoch in range(epochs):
    total_loss = 0
    for batch_x, _ in dataloader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        x_hat, mu, logvar = model(batch_x)
        loss, recon_loss, kld_loss = vae_loss(batch_x, x_hat, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataset):.4f}")

# -------------------------------
# 5. 生成新样本
# -------------------------------
num_samples = 10
z = torch.randn(num_samples, latent_dim).to(device)
generated = model.decode(z)
print("Generated samples:\n", generated.detach().cpu().numpy())

# -------------------------------
# 6. 评估指标示例
# -------------------------------
with torch.no_grad():
    x_hat, mu, logvar = model(X_tensor.to(device))
    mse = nn.MSELoss()(x_hat, X_tensor.to(device)).item()
    print(f"Reconstruction MSE on original data: {mse:.4f}")
    # 可以额外计算生成样本均值、方差和原数据对比
    gen_mean = generated.mean(0).cpu().numpy()
    gen_std = generated.std(0).cpu().numpy()
    orig_mean = X_tensor.mean(0).numpy()
    orig_std = X_tensor.std(0).numpy()
    print("Original mean:", orig_mean)
    print("Generated mean:", gen_mean)
    print("Original std:", orig_std)
    print("Generated std:", gen_std)


Epoch 1, Loss: 1.8057
Epoch 2, Loss: 1.5939
Epoch 3, Loss: 1.4477
Epoch 4, Loss: 1.4100
Epoch 5, Loss: 1.3604
Epoch 6, Loss: 1.3166
Epoch 7, Loss: 1.2575
Epoch 8, Loss: 1.2405
Epoch 9, Loss: 1.2154
Epoch 10, Loss: 1.1758
Epoch 11, Loss: 1.1411
Epoch 12, Loss: 1.1220
Epoch 13, Loss: 1.0996
Epoch 14, Loss: 1.0868
Epoch 15, Loss: 1.0670
Epoch 16, Loss: 1.0454
Epoch 17, Loss: 1.0452
Epoch 18, Loss: 0.9887
Epoch 19, Loss: 0.9899
Epoch 20, Loss: 0.9865
Epoch 21, Loss: 0.9733
Epoch 22, Loss: 0.9644
Epoch 23, Loss: 0.9500
Epoch 24, Loss: 0.9479
Epoch 25, Loss: 0.9540
Epoch 26, Loss: 0.9300
Epoch 27, Loss: 0.9085
Epoch 28, Loss: 0.9187
Epoch 29, Loss: 0.9098
Epoch 30, Loss: 0.9131
Epoch 31, Loss: 0.8982
Epoch 32, Loss: 0.8879
Epoch 33, Loss: 0.8879
Epoch 34, Loss: 0.9016
Epoch 35, Loss: 0.8809
Epoch 36, Loss: 0.8822
Epoch 37, Loss: 0.8818
Epoch 38, Loss: 0.8685
Epoch 39, Loss: 0.8711
Epoch 40, Loss: 0.8763
Epoch 41, Loss: 0.8509
Epoch 42, Loss: 0.8714
Epoch 43, Loss: 0.8586
Epoch 44, Loss: 0.86

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler


data = load_breast_cancer()
X = data['data']
X = StandardScaler().fit_transform(X)
X_tensor = torch.tensor(X, dtype=torch.float32)

dataset = TensorDataset(X_tensor, X_tensor)  # VAE输入=输出
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

input_dim = X_tensor.shape[1]
latent_dim = 5
target_dim = input_dim

# 超参数
eta_dec_sq = 1.0
eta_prior_sq = 1.0
beta = 2.0  # beta系数控制KL权重

# -------------------------------
# 2. Linear Variational Encoder & Decoder
# -------------------------------
class LinearVariationalEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.nn_mean = nn.Linear(input_dim, latent_dim, bias=False)
        self.nn_logvar = nn.Linear(input_dim, latent_dim, bias=True)

    def forward(self, x):
        batch_size = x.size(0)
        eps = torch.randn(batch_size, self.nn_mean.out_features, device=x.device)
        mu = self.nn_mean(x)
        logvar = self.nn_logvar(x)
        sigma = torch.exp(logvar / 2)
        z = mu + sigma * eps
        return {'z': z, 'mu': mu, 'sigma': sigma}

class LinearVariationalDecoder(nn.Module):
    def __init__(self, latent_dim, target_dim):
        super().__init__()
        self.l = nn.Linear(latent_dim, target_dim, bias=False)

    def forward(self, z):
        return self.l(z)

# -------------------------------
# 3. LinearBetaVAE
# -------------------------------
class LinearBetaVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, target_dim, eta_dec_sq, eta_prior_sq, beta):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.target_dim = target_dim
        self.eta_dec_sq = eta_dec_sq
        self.eta_prior_sq = eta_prior_sq
        self.beta = beta

        self.encoder = LinearVariationalEncoder(input_dim, latent_dim)
        self.decoder = LinearVariationalDecoder(latent_dim, target_dim)

    def forward(self, x, y):
        encoded = self.encoder(x)
        y_pred = self.decoder(encoded['z'])

        rec_loss = torch.square(y - y_pred).sum(-1).mean(0) / self.eta_dec_sq / 2

        kl_loss = 0.5 * (
            - torch.log(encoded['sigma']**2 / self.eta_prior_sq).sum(-1)
            - self.latent_dim
            + torch.norm(encoded['mu'], p=2, dim=-1)**2 / self.eta_prior_sq
            + (encoded['sigma']**2 / self.eta_prior_sq).sum(-1)
        ).mean(0)

        loss = rec_loss + self.beta * kl_loss

        return {
            'z': encoded['z'],
            'mu': encoded['mu'],
            'sigma': encoded['sigma'],
            'y_pred': y_pred,
            'loss': loss,
            'rec_loss': rec_loss,
            'kl_loss': kl_loss * self.beta,
            'enc_norm': self.encoder.nn_mean.weight.norm(),
            'dec_norm': self.decoder.l.weight.norm()
        }


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinearBetaVAE(input_dim, latent_dim, target_dim, eta_dec_sq, eta_prior_sq, beta).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 50
for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        out = model(batch_x, batch_y)
        out['loss'].backward()
        optimizer.step()
        total_loss += out['loss'].item() * batch_x.size(0)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataset):.4f}")


num_samples = 5
z = torch.randn(num_samples, latent_dim).to(device)
generated = model.decoder(z)
print("Generated samples:\n", generated.detach().cpu().numpy())


with torch.no_grad():
    out = model(X_tensor.to(device), X_tensor.to(device))
    mse = torch.mean(torch.square(out['y_pred'] - X_tensor.to(device))).item()
    print(f"Reconstruction MSE: {mse:.4f}")

    gen_mean = generated.mean(0).cpu().numpy()
    gen_std = generated.std(0).cpu().numpy()
    print("Generated mean:", gen_mean)
    print("Generated std:", gen_std)


Epoch 1, Loss: 25.4203
Epoch 2, Loss: 22.2783
Epoch 3, Loss: 20.9595
Epoch 4, Loss: 19.5449
Epoch 5, Loss: 19.0145
Epoch 6, Loss: 18.4434
Epoch 7, Loss: 17.6513
Epoch 8, Loss: 17.2449
Epoch 9, Loss: 16.6802
Epoch 10, Loss: 16.0857
Epoch 11, Loss: 15.6914
Epoch 12, Loss: 15.3926
Epoch 13, Loss: 15.0367
Epoch 14, Loss: 14.5037
Epoch 15, Loss: 14.4271
Epoch 16, Loss: 13.8683
Epoch 17, Loss: 13.8112
Epoch 18, Loss: 13.5293
Epoch 19, Loss: 13.4017
Epoch 20, Loss: 13.2370
Epoch 21, Loss: 13.3070
Epoch 22, Loss: 12.8002
Epoch 23, Loss: 13.0292
Epoch 24, Loss: 12.5734
Epoch 25, Loss: 12.6719
Epoch 26, Loss: 12.5672
Epoch 27, Loss: 12.2556
Epoch 28, Loss: 12.5799
Epoch 29, Loss: 12.3883
Epoch 30, Loss: 12.1475
Epoch 31, Loss: 12.3619
Epoch 32, Loss: 11.8727
Epoch 33, Loss: 11.9187
Epoch 34, Loss: 11.9313
Epoch 35, Loss: 11.8219
Epoch 36, Loss: 11.5998
Epoch 37, Loss: 11.6672
Epoch 38, Loss: 11.4977
Epoch 39, Loss: 11.6646
Epoch 40, Loss: 11.6519
Epoch 41, Loss: 11.5435
Epoch 42, Loss: 11.6207
E